# eCREAM Analysis 3: A model that must find nothing. 30-day mortality as a negative control

_The optional third experiment of the live demo, to be shown with its explanation. Both synthetic datasets. About ten minutes._

**Before you start.** The eCREAM federation holds the UC1 data model (265 variables in four
population blocks) and, today, two **synthetic** datasets of 5,000 subjects, one with dyspnoea
(`DYSP5K`) and one with transient loss of consciousness, TLoC (`TLOC5K`), generated in July 2026
from anonymized mockup samples. No real patient data are involved. Two properties of the
synthetic data shape every result: each cohort was generated from its own sample, so contrasts
*between* the datasets survive; each block of variables was resampled on its own, so
associations *across* blocks (age and death, saturation and death) do not. Known artefacts:
uniform triage codes and modes of arrival, door-to-nurse at zero, no "discharged" exit, ages
above 89 not capped.

**How this notebook works.** Every analysis is a *federated* call: the code selects the datasets
and the variables, defines a cohort with filters and asks the MIP for aggregates (counts, means,
a test, a model). No record ever reaches this notebook. Run the cells in order; each step says
what to look for and what result to expect. The same steps, for the web interface, are in
`docs/teaching_scenarios/`.

**What this analysis shows.** In real emergency data, age and hypoxaemia predict death; a
logistic regression of 30-day mortality on age, sex and the first oxygen saturation on room air
would find it. The synthetic datasets cannot show this: the generator resampled each block of
variables on its own (Demographics, Vital signs and Follow-up are three different blocks), so a
patient's oxygen reading no longer belongs to that patient's outcome. The model must find
nothing, and the demo shows that it does: **the platform reports no signal because there is
none.** Analyses 1 and 2 show that the platform can measure how emergency departments work;
this one shows that it does not invent findings.

**The cohort.** Both datasets with the filter *30-day mortality is Dead or Alive* and *Sex is
Male or Female*: 8,205 records, 864 deaths; the platform then removes the 477 records without an
oxygen saturation, so the model runs on 7,728 records with 811 deaths. Without the filter on
sex the platform cannot fit the model: the 29 records with an unknown sex contain no death
(complete separation); see the last cell.

## 1. Connect to the eCREAM federation and open the UC1 pathology

In [ ]:
import mip
from mip.filters import F

client = mip.Client.from_env()          # connection configured by the MIP portal
catalog = client.catalog()

try:
    dm = catalog.data_model("UC1")         # add version="eCREAMv8" if the catalogue holds several versions
except LookupError:
    display(catalog.summaries())           # the pathology label differs: pick it from this list
    raise
def dataset(code, label):
    # by the code, which is how the federation labels its datasets (seen 2026-10-12), else by the long label
    try:
        return dm.datasets[code]
    except (KeyError, LookupError):
        return dm.datasets[label]

dysp = dataset("DYSP5K", "Synthetic Dyspnea dataset (5000 subjects)")
tloc = dataset("TLOC5K", "Synthetic TLoC dataset (5000 subjects)")
DATASETS = [dysp, tloc]                   # every call of this notebook runs on both synthetic datasets
display(dm)
display(dm.datasets.to_frame())           # 21 dataset codes: four hold data today, 17 are the EDs' containers

In [ ]:
# The variable tree: the four population blocks of the UC1 case report form. The demo uses
# ALL_PATIENTS (logistics, demographics, door-to-event delays, dataset of origin) and, for the
# mortality model, two variables of DYSPNEA_AND_TLOC (vital signs, follow-up).
dm.tree(group="ALL_PATIENTS", include_variables=True)

## 2. Toolbox

In [ ]:
# ---- Toolbox: small helpers that turn MIP results into readable tables and charts. ----
# Run this cell once. You do not need to read it to follow the analysis; come back to it
# when you want to see how a result payload is unpacked.
import math
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from mip.preprocessing import MissingValuesHandler

POOLED = "all datasets"                 # the row describe() adds across the selected datasets
PCT = "{:.1f}"                          # number format of the percentage columns


def analysis(variables, filters=None, drop_missing=False, datasets=None):
    """A Pipeline on the datasets of this notebook (DATASETS, or the ones given) with the given
    variables and cohort filter. Tests and models need drop_missing=True: the platform requires an
    explicit missing-values step and analyses complete cases only. describe() keeps it off so it
    can count missing values."""
    handler = MissingValuesHandler(strategies={v: "drop" for v in variables}) if drop_missing else None
    return mip.Pipeline(analysis_set=dm.select(datasets=datasets or DATASETS, variables=variables),
                        filters=filters, handle_missing=handler)


def labels_of(variable):
    """{code: label} of a categorical variable, whatever format the catalogue uses."""
    raw = variable._data.get("enumerations") or {}
    if isinstance(raw, dict):
        return {str(k): str(v) for k, v in raw.items()}
    return {str(item.get("code") if isinstance(item, dict) else item):
            str(item.get("label") if isinstance(item, dict) else item) for item in raw}


def name_of(dataset):
    """Column name of a dataset in the tables: its code, or 'all datasets' for the pooled row."""
    return POOLED if dataset == POOLED else dataset._code


def rows_for(result, variable, dataset=POOLED):
    """The describe row of one variable: the pooled row by default, or the row of one dataset."""
    keys = (POOLED,) if dataset == POOLED else (dataset._code, dataset.label)
    return [r for r in result.raw.get("featurewise", [])
            if r.get("variable") == variable._code and r.get("dataset") in keys]


def numeric_by_dataset(result, variable, datasets=None):
    """n, missing, mean, SD, min, quartiles and max of a numerical variable, one row per dataset
    and the pooled row, from a describe result."""
    rows = []
    for ds in list(datasets or DATASETS) + [POOLED]:
        for r in rows_for(result, variable, ds):
            d = r.get("data") or {}
            rows.append({"dataset": name_of(ds), "n": d.get("num_dtps"), "missing": d.get("num_na"),
                         "mean": d.get("mean"), "sd": d.get("std"), "min": d.get("min"), "q1": d.get("q1"),
                         "median": d.get("q2"), "q3": d.get("q3"), "max": d.get("max")})
    return pd.DataFrame(rows)


def counts_by_dataset(result, variable, datasets=None):
    """Counts and percentages of the levels of a categorical variable, one column pair per dataset
    and for the pooled row; the number of missing values in .attrs['missing']."""
    lab = labels_of(variable)
    table, missing = {}, {}
    for ds in list(datasets or DATASETS) + [POOLED]:
        for r in rows_for(result, variable, ds):
            d = r.get("data") or {}
            counts = {str(k): v for k, v in (d.get("counts") or {}).items()}
            total = sum(counts.values())
            table[name_of(ds)] = pd.Series({label: counts.get(c, 0) for c, label in lab.items()})
            table[name_of(ds) + " %"] = pd.Series({label: 100 * counts.get(c, 0) / total if total else np.nan
                                                   for c, label in lab.items()})
            missing[name_of(ds)] = d.get("num_na")
    out = pd.DataFrame(table)
    out.index.name = variable.label
    out.attrs["missing"] = missing
    return out


def record_counts(filters, variable, datasets=None):
    """Records matching a filter, per dataset and pooled, from a describe of one variable that is
    never missing (the dataset of origin): the notebook's version of the filter counter."""
    res = analysis([variable], filters, datasets=datasets).describe()
    out = {}
    for ds in list(datasets or DATASETS) + [POOLED]:
        rows = rows_for(res, variable, ds)
        d = (rows[0].get("data") or {}) if rows else {}
        out[name_of(ds)] = d.get("num_total")          # None when fewer than 10 records match
    return out


def share_of(variable, level, filters, datasets=None):
    """% of the records with a given level (code) of a categorical variable among the records
    where it is recorded, pooled over the datasets, for a cohort filter."""
    res = analysis([variable], filters, datasets=datasets).describe()
    rows = rows_for(res, variable)
    counts = ((rows[0].get("data") or {}).get("counts") or {}) if rows else {}
    total = sum(counts.values())
    return (100 * counts.get(level, 0) / total if total else np.nan), total


def histograms_by_dataset(variable, bins=20, filters=None, datasets=None, unit=""):
    """One histogram per dataset (one federated call each), drawn side by side. Bins with fewer
    than 10 records come back empty: the platform's privacy rule."""
    datasets = list(datasets or DATASETS)
    fig, axes = plt.subplots(1, len(datasets), figsize=(6 * len(datasets), 3.4))
    for ax, ds in zip(np.atleast_1d(axes), datasets):
        h = analysis([variable], filters, drop_missing=True, datasets=[ds]).histogram(variable=variable, bins=bins).raw["histogram"][0]
        edges, counts = h["bins"], h["counts"]
        bars = [c or 0 for c in counts]
        ax.bar(edges[:-1], bars, width=np.diff(edges), align="edge", color="#4C72B0")
        hidden = sum(c is None for c in counts)
        peak = int(np.argmax(bars))
        ax.set_title(f"{variable.label}, {ds.label}")
        ax.set_xlabel(f"{variable.label} ({unit})" if unit else variable.label)
        ax.set_ylabel("records")
        print(f"{ds._code}: bins {edges[1] - edges[0]:.1f} wide, peak {bars[peak]} records at "
              f"{edges[peak]:.0f} to {edges[peak + 1]:.0f}, {hidden} bins under 10 records not drawn")
    plt.tight_layout()
    plt.show()


def fmt_p(p):
    if p is None or (isinstance(p, float) and np.isnan(p)):
        return "n/a"
    return "< 0.001" if p < 0.001 else f"{p:.3f}"


def ttest_summary(result):
    """Independent t-test result -> t, p, df, mean difference with its 95% CI, Cohen's d, n."""
    raw = result.raw if isinstance(result.raw, dict) else {}
    return {k: raw.get(k) for k in ("t_stat", "p", "df", "mean_diff", "ci_lower", "ci_upper", "cohens_d", "n_obs")}


def show_ttest(result, what, unit=""):
    # one line with the t-test result; a key the platform does not return is shown as n/a
    s = ttest_summary(result)
    num = lambda v, spec: format(v, spec) if isinstance(v, (int, float)) else "n/a"
    print(f"{what}: difference of means {num(s['mean_diff'], '.1f')} {unit} (95% CI {num(s['ci_lower'], '.1f')} "
          f"to {num(s['ci_upper'], '.1f')}), t = {num(s['t_stat'], '.1f')} on {num(s['df'], '.0f')} df, "
          f"p = {fmt_p(s['p'])} ({num(s['p'], '.1e')}), Cohen's d {num(s['cohens_d'], '.2f')}, n = {num(s['n_obs'], ',')}")
    missing = [k for k, v in s.items() if v is None]
    if missing:
        print(f"   (keys the platform did not return: {', '.join(missing)}; keys returned: {', '.join(result.raw)})")
    return s


def chi2_summary(result, x, y):
    """Chi-squared Test result -> chi2, p, dof, and the expected counts as a table with labels."""
    raw = result.raw if isinstance(result.raw, dict) else {}
    lab_x, lab_y = labels_of(x), labels_of(y)
    expected = pd.DataFrame(raw.get("expected") or [],
                            index=[lab_x.get(str(i), str(i)) for i in raw.get("x_labels") or []],
                            columns=[lab_y.get(str(c), str(c)) for c in raw.get("y_labels") or []])
    expected.index.name, expected.columns.name = x.label, y.label
    return raw.get("chi2"), raw.get("p_value", raw.get("p")), raw.get("dof"), expected


def term_label(term, variables):
    """Model term -> readable label: 'sex[2]' -> 'Sex = Female', 'age' -> 'Age (per unit)'."""
    by_code = {v._code: v for v in variables}
    term = str(term)
    if "[" in term and term.endswith("]"):
        base, level = term[:-1].split("[", 1)
        v = by_code.get(base)
        return f"{v.label} = {labels_of(v).get(level, level)}" if v else term
    v = by_code.get(term)
    return f"{v.label} (per unit)" if v else term


def odds_ratio_table(result, variables):
    """Logistic regression result -> one row per term with coefficient, odds ratio, 95% CI and p.
    Categorical covariates appear as 'variable = level' relative to the reference level
    (the first code of the variable) that the platform leaves out."""
    frame = result.to_frame().copy()
    frame = frame[~frame["feature"].astype(str).str.lower().isin(["intercept", "const"])]
    frame["term"] = frame["feature"].map(lambda t: term_label(t, variables))
    for col in ("coefficient", "odds_ratio", "or_lower_ci", "or_upper_ci", "p"):
        if col not in frame.columns:
            frame[col] = np.nan
    out = frame[["term", "coefficient", "odds_ratio", "or_lower_ci", "or_upper_ci", "p"]]
    return out.rename(columns={"or_lower_ci": "ci_low", "or_upper_ci": "ci_high"}).reset_index(drop=True)


def show_odds_ratios(table, title):
    """Table and forest plot (log scale) of odds ratios."""
    display(table.style.format({"coefficient": "{:.4f}", "odds_ratio": "{:.3f}", "ci_low": "{:.3f}",
                                "ci_high": "{:.3f}", "p": "{:.3f}"}).hide(axis="index"))
    t = table.iloc[::-1]
    fig, ax = plt.subplots(figsize=(7, 0.6 * len(t) + 1.5))
    ax.errorbar(t["odds_ratio"], range(len(t)),
                xerr=[t["odds_ratio"] - t["ci_low"], t["ci_high"] - t["odds_ratio"]], fmt="o", capsize=4)
    ax.axvline(1, linestyle="--", linewidth=1)
    ax.set_xscale("log")
    ax.set_yticks(range(len(t)))
    ax.set_yticklabels(t["term"])
    ax.set_xlabel("odds ratio (log scale)")
    ax.set_title(title)
    plt.tight_layout()
    plt.show()


def roc_and_matrix(cv, event="death", non_event="survivor"):
    """Cross-validation result -> the ROC curve of every fold with its area, and the confusion
    matrix summed over the folds at the platform's 50% cut-off."""
    curves, cm = cv.raw["roc_curves"], cv.raw["confusion_matrix"]
    aucs = [c["auc"] for c in curves]
    print("Area under the ROC curve, per fold:", ", ".join(f"{a:.2f}" for a in aucs), f"| average {np.mean(aucs):.2f}")
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.6))
    for c in curves:
        axes[0].plot(c["fpr"], c["tpr"], linewidth=1, label=f"{c['name'].replace('_', ' ')}: area {c['auc']:.2f}")
    axes[0].plot([0, 1], [0, 1], linestyle="--", color="grey", label="coin toss: area 0.50")
    axes[0].set_xlabel(f"share of {non_event}s wrongly called {event}s")
    axes[0].set_ylabel(f"share of {event}s caught")
    axes[0].set_title("ROC curve, on patients the model had not seen")
    axes[0].legend(fontsize=8, loc="lower right")
    matrix = np.array([[cm["tn"], cm["fp"]], [cm["fn"], cm["tp"]]])  # rows: the truth; columns: the verdict
    axes[1].imshow(matrix, cmap="Blues")
    for i in range(2):
        for j in range(2):
            axes[1].text(j, i, f"{matrix[i, j]:,}", ha="center", va="center", fontsize=14,
                         color="white" if matrix[i, j] > matrix.max() / 2 else "black")
    axes[1].set_xticks([0, 1])
    axes[1].set_xticklabels([f"called a {non_event}", f"called a {event}"])
    axes[1].set_yticks([0, 1])
    axes[1].set_yticklabels([f"was a {non_event}", f"was a {event}"])
    axes[1].set_title("Confusion matrix at the platform's 50% cut-off")
    plt.tight_layout()
    plt.show()
    n = sum(cm.values())
    sens = 100 * cm["tp"] / (cm["tp"] + cm["fn"]) if cm["tp"] + cm["fn"] else float("nan")
    spec = 100 * cm["tn"] / (cm["tn"] + cm["fp"]) if cm["tn"] + cm["fp"] else float("nan")
    print(f"Accuracy {100 * (cm['tp'] + cm['tn']) / n:.1f}%, sensitivity {sens:.1f}%, specificity {spec:.1f}%")
    return aucs


def saved_experiments():
    """The experiments saved from the web interface (My experiments), as the platform lists them."""
    items = client.experiments().list()
    if not items:
        print("No saved experiment is listed (nothing saved yet, or the registry answers differently on this platform).")
        return pd.DataFrame()
    return pd.DataFrame(items)


print("Toolbox ready.")

## 3. The datasets and the variables of this analysis

| Code | Label in the MIP | Tree location (under UC1) | Role | Values |
|---|---|---|---|---|
| `dataset` | Dataset Registry origin | ALL_PATIENTS / Dataset Registry Origin | which cohort a record belongs to; both selected | 21 codes, listed in the tree |
| `mortality_30_day` | 30-day mortality | DYSPNEA_AND_TLOC / Follow-up | outcome of the Logistic Regression, positive class Dead | 1 Dead; 2 Alive; 3 Unknown |
| `age` | Age | ALL_PATIENTS / Demographics | predictor | integer |
| `sex` | Sex | ALL_PATIENTS / Demographics | predictor; cohort filter Male or Female | 1 Male; 2 Female; 3 Unknown |
| `spo2_AA_first` | SpO2 room air (1st) | DYSPNEA_AND_TLOC / Clinical Examination / Vital Signs | predictor: first oxygen saturation on room air | integer, % |

Variables are picked by their **label**, as the tree shows it; the tree location tells you where
to find them in `dm.tree()` and in the MIP interface. The code (for example `door_doctor_delay`)
is what the csv column is called; you will see it in some result payloads. Both synthetic
datasets are selected for every call.

In [ ]:
mortality  = dm.variables["30-day mortality"]
age        = dm.variables["Age"]
sex        = dm.variables["Sex"]
spo2       = dm.variables["SpO2 room air (1st)"]
dataset    = dm.variables["Dataset Registry origin"]      # which cohort a record belongs to
cohort = F(mortality).isin(["1", "2"]) & F(sex).isin(["1", "2"])   # known outcome, known sex
display(mip.to_frame([mortality, age, sex, spo2, dataset]))

## Step 1. The outcome and the predictors, descriptively

No filter yet: what the two datasets hold.

*Expected, 30-day mortality:* dyspnoea 435 dead, 3,674 alive, 161 unknown, 730 missing; TLoC
429, 3,694, 155, 722; dead among dead or alive 10.6% and 10.4%. *SpO2 on room air:* dyspnoea
4,420 measured (580 missing), mean 90.2 (SD 8.0), median 91, 59 to 100; TLoC 5,000 measured,
mean 97.2 (SD 1.4), median 97, 92 to 100; all datasets mean 93.9. One patient in ten dies
within 30 days, the same in both cohorts; the saturations are plausible per cohort (hypoxic
breathless patients, normal fainting patients).

In [ ]:
review = analysis([mortality, spo2, age, sex]).describe()
outcome = counts_by_dataset(review, mortality)
display(outcome.style.format({c: PCT for c in outcome.columns if c.endswith("%")}))
print("missing:", outcome.attrs["missing"])
dead_of_known = {ds: 100 * outcome.loc["Dead", ds] / (outcome.loc["Dead", ds] + outcome.loc["Alive", ds])
                 for ds in (name_of(dysp), name_of(tloc), POOLED)}
print("dead, of dead + alive:", {k: f"{v:.1f}%" for k, v in dead_of_known.items()})
display(numeric_by_dataset(review, spo2).style.format({c: "{:.1f}" for c in ("mean", "sd")}).hide(axis="index"))

## Step 2. The filter

*Expected:* 8,205 of the 10,000 records match (4,094 dyspnoea, 4,111 TLoC); 864 deaths among
them.

In [ ]:
print("records matching the filter:", record_counts(cohort, dataset))
pct, n = share_of(mortality, "1", cohort)
print(f"dead: {pct:.1f}% of {n:,} records with a known outcome and a known sex")

## Step 3. A descriptive look that already says it all

The share of deaths in bands of saturation, age and sex, under the cohort filter (one describe
per band), and the Chi-squared Test of mortality by sex.

*Expected, share dead:* saturation below 90: 10.0%; 90 to 94: 12.6%; 95 or more: 10.1%. Age
under 65: 10.7%; 65 to 79: 10.5%; 80 and over: 10.4%. Men 10.4%, women 10.7% (chi-square 0.08,
p = 0.78). Flat: a real emergency cohort would show mortality climbing with age and with
hypoxaemia; here every band dies at one in ten.

In [ ]:
bands = {"SpO2 below 90": F(spo2) < 90, "SpO2 90 to 94": (F(spo2) >= 90) & (F(spo2) < 95), "SpO2 95 or more": F(spo2) >= 95,
         "age under 65": F(age) < 65, "age 65 to 79": (F(age) >= 65) & (F(age) < 80), "age 80 and over": F(age) >= 80,
         "men": F(sex) == "1", "women": F(sex) == "2"}
rows = []
for label, f in bands.items():
    pct, n = share_of(mortality, "1", cohort & f)
    rows.append({"band": label, "records": n, "% dead": pct})
flat = pd.DataFrame(rows)
display(flat.style.format({"% dead": PCT}).hide(axis="index"))
chi = analysis([mortality, sex], cohort, drop_missing=True).chi_square_test(x=sex, y=mortality)
chi2, p, dof, _ = chi2_summary(chi, sex, mortality)
print(f"30-day mortality by sex: chi2 = {chi2:.2f}, p = {fmt_p(p)}")
ax = flat.set_index("band")["% dead"].plot(kind="barh", color="#C44E52", figsize=(7, 3.4))
ax.set_xlabel("% dead within 30 days"); ax.set_ylabel(""); ax.set_xlim(0, 20); ax.set_title("A flat picture")
plt.tight_layout()
plt.show()

## Step 4. The model: Logistic Regression

Outcome 30-day mortality with the positive class Dead, predictors age, sex and saturation,
under the cohort filter.

*Expected (7,728 records, 811 deaths):* Sex = Female odds ratio 1.03 (95% CI 0.89 to 1.20,
p = 0.65) against men; Age 1.000 per year (0.996 to 1.003, p = 0.93); SpO2 1.004 per point
(0.992 to 1.016, p = 0.52). McFadden's pseudo R-squared 0.0001. Nothing predicts anything; the
model is the intercept.

In [ ]:
predictors = [age, sex, spo2]
t0 = time.perf_counter()
model = analysis([mortality] + predictors, cohort, drop_missing=True).logistic_regression(x=predictors, y=mortality, positive_class="1")
summary = model.raw["summary"]
print(f"{summary['n_obs']:,} records in the model; the federation answered in {time.perf_counter() - t0:.1f} s")
show_odds_ratios(odds_ratio_table(model, predictors), "Odds of dying within 30 days, synthetic cohorts")
print(f"McFadden's pseudo R-squared {summary.get('r_squared_mcf', float('nan')):.5f}; log-likelihood {summary.get('ll', float('nan')):.1f} "
      f"against {summary.get('ll0', float('nan')):.1f} with the intercept alone")

## Step 5. Cross-validation: the ROC curve and the confusion matrix

The same model, fitted on two thirds of the records and tested on the remaining third, three
times (Logistic Regression Cross-validation, 3 folds).

*Expected:* areas under the ROC curve between 0.48 and 0.51 in every fold, 0.49 to 0.50 on
average, with three folds as with five (the exact values depend on the order in which the
records fall into the folds; the local stand-in gave 0.50, 0.50 and 0.49). Confusion matrix: 6,917 survivors called survivors, 811 deaths called
survivors, nobody called dead; accuracy 89.5%, sensitivity 0%, specificity 100%. A coin flip:
the model has nothing to rank patients by; the highest predicted risk in any fold is 14%, so at
the 50% cut-off every patient is called a survivor, and the model is "right" 89.5% of the time
only because 89.5% survive. When the event is rare, accuracy misleads: read the area under the
curve and the sensitivity. The honest fingerprint of synthetic data, and a negative control for
the platform.

In [ ]:
cv = analysis([mortality] + predictors, cohort, drop_missing=True).logistic_regression_cv(x=predictors, y=mortality, positive_class="1", n_splits=3)
aucs = roc_and_matrix(cv, event="death", non_event="survivor")
cv5 = analysis([mortality] + predictors, cohort, drop_missing=True).logistic_regression_cv(x=predictors, y=mortality, positive_class="1", n_splits=5)
print("With five folds, areas:", ", ".join(f"{c['auc']:.2f}" for c in cv5.raw["roc_curves"]))

## Step 6. State the result

A model answer:

> On 7,728 synthetic patients with a known 30-day outcome, age, sex and oxygen saturation did
> not predict death (all odds ratios 1.0; cross-validated area under the ROC curve near 0.5). This
> is expected: the synthetic generator resampled the demographic, vital-sign and follow-up
> blocks independently, so no association across blocks exists in these data. The platform
> found nothing because there was nothing to find; on real data the same experiment is
> expected to show age and hypoxaemia as predictors.

## What happens without the filter on sex

The 29 records with Sex = Unknown contain no death: the term Sex[Unknown] has no finite estimate
(complete separation). Depending on the order of the records, a fit stops with an error (a singular matrix) or
returns a coefficient near -17 with a standard error in the thousands; the cell catches either
outcome, for the plain regression and for the cross-validation.

In [ ]:
known_outcome = F(mortality).isin(["1", "2"])             # the outcome filter alone
try:
    loose = analysis([mortality] + predictors, known_outcome, drop_missing=True).logistic_regression(x=predictors, y=mortality, positive_class="1")
    display(odds_ratio_table(loose, predictors).style.format({"coefficient": "{:.2f}", "odds_ratio": "{:.3g}", "ci_low": "{:.3g}", "ci_high": "{:.3g}", "p": "{:.3f}"}).hide(axis="index"))
    print("Sex = Unknown: a coefficient without a finite estimate, as expected")
except Exception as error:
    print("The platform refused or failed the fit, as expected:", error)
try:
    analysis([mortality] + predictors, known_outcome, drop_missing=True).logistic_regression_cv(x=predictors, y=mortality, positive_class="1", n_splits=3)
    print("The cross-validation ran through: the fits converged on these folds despite the separation.")
except Exception as error:
    print("The cross-validation failed, as expected:", error)